# Running heavy work in the JupyDo job queue (R)

Your environment has limited CPU and RAM. Heavy work goes to the **job queue**: put `%%job` on the
first line of a cell and the cell runs as a **batch job**, with the resources you ask for, when they are free.

How a job works:
- it runs in a **new container**, made from a snapshot of this environment taken when you submit it:
  same software (also what you installed yourself) and the same `/sharedFolder`;
- it runs the cell **alone**: variables of this notebook are **not** available in the job;
- only the **files it writes** survive: save your results in `/sharedFolder` (the job starts in this notebook's folder);
- `--time` is how long you expect it to run: if others are waiting, a job running longer is stopped.

Options: `--time` (e.g. `30m`, `2h`), `--cpus`, `--mem` (GB), `--gpus`, `--name`.

## 1. Queue status
`%jobs` (alone in a cell) lists your jobs and the free resources of the queue.

In [ ]:
%jobs

## 2. A first job
The cell runs with `Rscript`: it prints a message, waits 30 seconds and ends. Run the cell: it returns at once with the job id.

In [ ]:
%%job --time 5m --cpus 1 --mem 2 --name hello_r
cat('Hello from the job container', Sys.info()[['nodename']], '\n')
Sys.sleep(30)
cat('done\n')

## 3. Follow it
`%jobs` shows its state. `%job_log <id>` (alone in a cell) shows what it printed: replace `<id>` with the id printed above.

In [ ]:
%jobs

In [ ]:
# %job_log <id>

## 4. A job that saves a result
The job computes something and **writes it to a file** (`saveRDS`). When it is done, read it here with `readRDS`.

In [ ]:
%%job --time 10m --cpus 2 --mem 4 --name model_r
set.seed(1)
x <- rnorm(1e6)
y <- 3 * x + rnorm(1e6)
fit <- lm(y ~ x)
dir.create('results', showWarnings = FALSE)
saveRDS(coef(fit), 'results/coefficients.rds')
print(coef(fit))

In [ ]:
# when %jobs says 'done':
readRDS('results/coefficients.rds')

## 5. Cancel a job
`%job_cancel <id>` (alone in a cell) cancels a pending job or stops a running one.

In [ ]:
# %job_cancel <id>